# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Refresh / Content Opportunity Scoring**

I want to explore whether observable search, traffic, freshness and engagement signals can be used to help prioritise which content pages to review first. I selected this lane because it brings together the available data and onto a human decision – a content team has only a finite amount of time, and it needs to know which pages to pay attention to first. Then I'm going to take a step back and start with simple, explainable signals and a clear baseline and determine if a machine-learning model offers enough value to warrant the complexity. This is a provisional selection and I may change the question based on what I learn about the data in the warehouse and validation requirements.




In [6]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("/data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Run this notebook from the repository root."
    )

df = pd.read_csv(DATA_PATH)
print(f"Starter dataset shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print("Required columns present:", all(c in df.columns for c in ["content_id", "client_id"]))

Starter dataset shape: 30,000 rows × 44 columns
Required columns present: True


## 2. The question: decision, action, cost of a wrong call

Research question: Is it possible to rank content pages based on signals that can be observed to determine which pages should be considered first for enhancement?
When there is limited review capacity, which pages of content should the content team review first?
Who is going to take action on it: A human content/SEO reviewer makes the call about what needs to be explored in the ranked list.

Potential actions: Based on the evidence, a content reviewer may consider a refresh, CTR/snippet improvement, engagement issue, continued monitoring and more.

Cost of a wrong call: If a false positive is made, then review time can be wasted or a content change will be requested. If the page is found to be unimportant, or the problem is not found, because of a false negative, this can happen. The system will therefore be used as a decision support system, rather than as an automatic guide to change/replace a page.

In [7]:
print("Unique content pages:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

if "trend_direction" in df.columns:
    print("Trend values:")
    print(df["trend_direction"].value_counts(dropna=False))
else:
    print("trend_direction is not present in this starter file.")

Unique content pages: 30000
Unique clients: 32
Trend values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Quick look at the data (2-3 real numbers)

The following code uses the starter CSV to determine the supporting numbers, rather than being hard coded in. The numbers listed are not model results but are used only to determine if the lane is worth investigating.


In [8]:

required = ["content_id", "impressions_90d", "content_age_days", "trend_direction"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected starter columns: {missing}")

eligible = df.loc[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].drop_duplicates(subset="content_id")

eligible_pages = len(eligible)
declining_pages = int((eligible["trend_direction"] == "down").sum())
declining_pct = 100 * declining_pages / eligible_pages if eligible_pages else 0

print(f"Eligible pages for the starter refresh problem: {eligible_pages:,}")
print(f"Eligible pages with a downward trend: {declining_pages:,}")
print(f"Share of eligible pages with a downward trend: {declining_pct:.1f}%")


Eligible pages for the starter refresh problem: 30,000
Eligible pages with a downward trend: 16,262
Share of eligible pages with a downward trend: 54.2%


## What these numbers tell me
Starter data includes a meaningful set of eligible content pages, and the observed trends in these pages were measured. With pages going down, it is a useful problem to investigate to be able to prioritize. The current data, however, does not prove that a downward trend will lead to recovery, but merely indicates a downward trend. The following steps will involve determining if a transparent ranking can be used to suggest candidates for useful reviews and if a learned method can enhance that ranking.

## 4. Careful words: what I can and can't claim

What I can assert: I can measure patterns and relationships in the data that is available for observation. I can evaluate the ability of various signals to prioritize pages and the performance of a ranking method over a transparent baseline, using an appropriate validation design. The end product will be a directional evidence-based decision support for human review.

What I cannot claim: If traffic or rankings improved, I can't claim it was due to a refresh, unless an appropriate experiment or causal design backs it up. I'm not a Google algorithm discoverer or foreteller. I also don't consider a high model score as a sign that a page will recover after it is changed.

Validation principle: As the project converts to the future, features should be based on information before the decision point and not be in the future target window. I will also look for time or client specific leakage before using the model results.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.